## Import Libraries

In [1]:
import os
import numpy as np
import splitfolders
import seaborn as sns
import tensorflow as tf
import albumentations as A
import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix
from tensorflow.keras import layers,models
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import image_dataset_from_directory


c:\Users\Abdulla\AppData\Local\Programs\Python\Python313\Lib\site-packages\albumentations\check_version.py:147: UserWarning: Error fetching version info <urlopen error _ssl.c:1011: The handshake operation timed out>
  data = fetch_version_info()


## Split Data

In [ ]:
dataset = "C:\\Users\\Abdulla\\OneDrive - Faculty of Computers and Information\\Desktop\\Data"

classes = os.listdir(dataset)
num_classes = len(classes)
print(f"Number of classes: {num_classes}")
print(f"Classes: {classes}")
    
# splitfolders.ratio(dataset,output='dataset',seed=1337,ratio=(.7,.2,.1))

# sample = tf.keras.utils.image_dataset_from_directory("dataset/train",image_size=(256,256),batch_size=32)

# train_ds = image_dataset_from_directory("dataset/train",image_size=(256,256),batch_size=None)
# val_ds = image_dataset_from_directory("dataset/val",image_size=(256,256),batch_size=None,shuffle=False)
# test_ds = image_dataset_from_directory("dataset/test",image_size=(256,256),batch_size=None,shuffle=False)

Number of classes: 50
Classes: ['1', '2', '3', 'Anubis Shrine', 'bust of the Greco-Egyptian god Serapis', 'Ceremonial Golden Throne of King Tutankhamun', 'coffin of King Psusennes I,', 'Coffins of Tutankhamun', 'Colossal Osiride Statue Head of Queen Hatshepsut', 'Column of King Merneptah', 'depicting the pharaoh Akhenaten', 'Emperor Augustus', 'Emperor Hadrian', 'goddess Isis', 'Hanging Obelisk of Ramesses II', 'Head of the god Serapis', 'khafre enthroned', 'King Ramesses III between the deities Horus and Seth', 'mask of King Psusennes I', 'Mask of Tutankhamun', 'Mitri', 'mosaic portrait of Queen Berenice II', 'Narmer Palette', 'obelisk of Queen Hatshepsut', 'Pharaoh Akhenaten', 'Poetical Stela of Thutmose III,', 'Prince Rahotep and his wife Nofret', 'ptah ramses sekhmet', 'Ramesses II as a child\xa0', 'Roman Emperor Marcus Aurelius', 'Roman Emperor Septimius Severus', 'shrines of Tutankhamu', 'Solar Boat of King Khufu', 'Sphinx of King Amenemhat III', 'Sphinx of Queen Hatshepsut', 'Sp

## Visualize Sample

In [ ]:
plt.figure(figsize=(9,9))

for images, labels in sample.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i+1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(classes[int(labels[i])])
        plt.axis("off")
    
plt.show()

## Data Augmentation

In [ ]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Rotate(limit=15, p=0.5),
    A.Blur(p=0.2),
    A.GaussNoise(var_limit=(10.0, 50.0),p=0.2),
    A.HueSaturationValue(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transform = A.Compose([
    A.Resize(height=256, width=256),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

## Apply Augmentation

In [ ]:
def apply_tranform(image,label,transform):
    def apply_augmentation(image):
        if len(image.shape) == 4:
            image = image[0]
            
        augmented = transform(image=image)
        return augmented['image']
        
    
    augmented_image = tf.numpy_function(apply_augmentation,[image],tf.float32)
    
    augmented_image.set_shape((256,256,3))
    return augmented_image,label

main_train_ds = train_ds.map(lambda image,label:apply_tranform(image, label, train_transform)).batch(32).shuffle(1000).prefetch(tf.data.AUTOTUNE)
main_val_ds = val_ds.map(lambda image,label:apply_tranform(image, label, val_transform)).batch(32).shuffle(1000).prefetch(tf.data.AUTOTUNE)
main_test_ds = test_ds.map(lambda image,label:apply_tranform(image, label, val_transform)).batch(32).shuffle(1000).prefetch(tf.data.AUTOTUNE)

## Visualize Augmented Data

In [ ]:
for images, labels in main_train_ds.take(1):
    plt.Figure(figsize=(10,10))
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(classes[int(labels[i])])
    plt.axis("off")
    plt.show()

## CNN Model

In [ ]:
cnn = models.Sequential([
    layers.Conv2D(32,(3,3), activation='relu', input_shape=(256,256,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(64,(3,3), activation='relu' ),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(128,(3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(256,(3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),
    
    layers.GlobalAveragePooling2D(),
        
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    
    layers.Dense(num_classes, activation='softmax')
])

cnn.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

cnn.summary()

## Fit Model

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

cnn.fit(main_train_ds,
        validation_data=main_val_ds,
        epochs=30,
        callbacks=[early_stopping])

In [ ]:
# cnn.fit(
#     main_train_ds,
#     validation_data=main_val_ds,
#     epochs=50,
#     initial_epoch=30
# )

## Visualize Confusion Matrix

In [ ]:
y_pred = cnn.predict(main_test_ds)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.concatenate([y for x, y in main_test_ds], axis=0)

cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="rocket",
    xticklabels=classes,
    yticklabels=classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()